## Extract data

In [105]:
import pandas as pd
import numpy as np

In [106]:
import os
print(os.getcwd())

/Users/Kirill/Documents/GitHub/Data-Warehouse


In [107]:
# # Ensure the required library is installed 
# (Can we do it (openpyxl)? Will it be a problem?)
# %pip install openpyxl

# Read the Excel file
file_path = "bitre_fatalities_dec2024.xlsx"
df = pd.read_excel(file_path, sheet_name="BITRE_Fatality", skiprows=4) # Can be improved
print(df.head())


   Crash ID State  Month  Year Dayweek      Time Crash Type Bus Involvement  \
0  20241115   NSW     12  2024  Friday  04:00:00     Single              No   
1  20241125   NSW     12  2024  Friday  06:15:00     Single              No   
2  20246013   Tas     12  2024  Friday  09:43:00   Multiple              No   
3  20241002   NSW     12  2024  Friday  10:35:00   Multiple              No   
4  20242261   Vic     12  2024  Friday  11:30:00   Multiple              -9   

  Heavy Rigid Truck Involvement Articulated Truck Involvement  ... Age  \
0                            No                            No  ...  74   
1                            No                            No  ...  19   
2                            No                            No  ...  33   
3                            No                            No  ...  32   
4                            -9                            -9  ...  62   

  National Remoteness Areas                           SA4 Name 2021  \
0  Inner 

In [108]:
# Clean the columnnames
# Remove leading and trailing whitespace, convert to lowercase, and replace spaces with underscores

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

# change sa4_name_2021 to sa4_name, national_lga_name_2021 to lga_name
df.rename(columns={
    'national_remoteness_areas': 'remoteness_areas',
    'sa4_name_2021': 'sa4_name',
    'national_lga_name_2021': 'lga_name',
    'national_road_type': 'road_type'
}, inplace=True)

In [109]:
# Add a serial number for each person killed in the accident
df['victim_number'] = df.groupby('crash_id').cumcount() + 1

# move the victim_number column to the front
cols = df.columns.tolist()
cols.insert(1, cols.pop(cols.index('victim_number')))
df = df[cols]
print(df.head())



   crash_id  victim_number state  month  year dayweek      time crash_type  \
0  20241115              1   NSW     12  2024  Friday  04:00:00     Single   
1  20241125              1   NSW     12  2024  Friday  06:15:00     Single   
2  20246013              1   Tas     12  2024  Friday  09:43:00   Multiple   
3  20241002              1   NSW     12  2024  Friday  10:35:00   Multiple   
4  20242261              1   Vic     12  2024  Friday  11:30:00   Multiple   

  bus_involvement heavy_rigid_truck_involvement  ... age  \
0              No                            No  ...  74   
1              No                            No  ...  19   
2              No                            No  ...  33   
3              No                            No  ...  32   
4              -9                            -9  ...  62   

           remoteness_areas                                sa4_name  \
0  Inner Regional Australia                                Riverina   
1  Inner Regional Australia 

In [110]:
# Save the cleaned DataFrame to a new Excel file
# output_file_path = "bitre_fatalities_cleaned.xlsx"
# df.to_excel(output_file_path, index=False)
# print(f"Cleaned data saved to {output_file_path}")


## Data transformation

Data transformation to apply:

1. dayweek: Drop
2. time: Categorise into rush time and usual time:
    Rush hours: 
    Morning Peak:
    Typically between 7 am and 9 am, as commuters head to work or school. 

    Evening Peak:
    Typically between 4 pm and 6 pm, as commuters travel home from work or school. 

    Not holiday, not weekend

Can be improved according to the state, city and so on

3. bus_involvement, heavy_rigid_truck_involvement, articulated_truck_involvement - treat -9 missing values
4. speed_limit: Categorise as follows:
    For all except NT:
        0-40 - low
        41-50 - med
        51-80 - high
        81 - inf - very high
    
    For NT:
        0-40 - low
        41-60 - med
        61-80 - high
        81 - inf - very high

    treat -9 as missing value
5. road_user:
    treat Other/-9, Unknown - as missing value

6. gender:
    treat -9 - as missing value

7. age: drop

8. national_remoteness_areas:
    treat Unknown - as missing value

9. sa4_name_2021:
    treat Unknown, Blank - as missing value

10. national_lga_name_2021:
    treat Unknown, Blank - as missing value

11. national_road_type:
    treat Undetermined - as missing value

12. christmas_period, easter_period:
    transform into is_holiday

13. age_group:
    treat -9 - as missing value

14. day_of_week:
    treat Unknown - as missing value

15. time_of_day:
    treat Unknown - as missing value

In [111]:
# Create new variable 'holiday' base on christmas_period, easter_period. If either is true, then holiday = 1, else 0
df['holiday'] = np.where(
    (df['christmas_period'].fillna(0) == "Yes") | (df['easter_period'].fillna(0) == "Yes"),
    "Yes",
    "No"
)


In [112]:
# convert the 'time' column to datetime format
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.time

# categorize time of day by rush hous:
# For all holiday == "No", day_of_week == "Weekday" set rush: 
# 07:00:00 - 09:00:00 = "Rush"
# 16:00:00 - 18:00:00 = "Rush"
Weekday = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
df['time'] = np.where(
    (df['holiday'] == "No") & (df['dayweek'].isin(Weekday)) & (
        ((df['time'] >= pd.to_datetime("07:00:00", format='%H:%M:%S').time()) & (df['time'] <= pd.to_datetime("09:00:00", format='%H:%M:%S').time())) |
        ((df['time'] >= pd.to_datetime("16:00:00", format='%H:%M:%S').time()) & (df['time'] <= pd.to_datetime("18:00:00", format='%H:%M:%S').time()))
    ),
    "Rush",
    "Not Rush"
)
print(df['time'].unique())


['Not Rush' 'Rush']


In [113]:
# set  NaN values for all 'Other/-9', '-9', 'Unknown', 'Undetermined' in all columns
nan_values = ['Other/-9', '-9', 'Unknown', 'Undetermined', -9]
df.replace(nan_values, np.nan, inplace=True)
print(df.head())

   crash_id  victim_number state  month  year dayweek      time crash_type  \
0  20241115              1   NSW     12  2024  Friday  Not Rush     Single   
1  20241125              1   NSW     12  2024  Friday  Not Rush     Single   
2  20246013              1   Tas     12  2024  Friday  Not Rush   Multiple   
3  20241002              1   NSW     12  2024  Friday  Not Rush   Multiple   
4  20242261              1   Vic     12  2024  Friday  Not Rush   Multiple   

  bus_involvement heavy_rigid_truck_involvement  ...  \
0              No                            No  ...   
1              No                            No  ...   
2              No                            No  ...   
3              No                            No  ...   
4             NaN                           NaN  ...   

           remoteness_areas                                sa4_name  \
0  Inner Regional Australia                                Riverina   
1  Inner Regional Australia  Sydney - Baulkham Hills

In [114]:
# check for right data type 
try :
    df['speed_limit'] = df['speed_limit'].astype(float)
except ValueError:
    # If conversion to int fails, print values that cannot be converted
    print("Values that cannot be converted to int:")
    print(df[~df['speed_limit'].apply(lambda x: isinstance(x, int) or pd.isna(x))]['speed_limit'].unique())



Values that cannot be converted to int:
['<40']


<40 means less than 40, which is 'low' speed category
just set this value to 40

In [115]:
# set '<40' speed_limit to 40
df['speed_limit'] = df['speed_limit'].replace('<40', 40)

In [116]:
# speed_limit: Categorise as follows:
#     For all except NT:
#         0-40 - low
#         41-50 - med
#         51-80 - high
#         81 - inf - very high
    
#     For NT:
#         0-40 - low
#         41-60 - med
#         61-80 - high
#         81 - inf - very high

df['speed_limit'] = np.where(
    df['state'] != "NT",
    np.select(
        [
            (df['speed_limit'] > 0) & (df['speed_limit'] <= 40),
            (df['speed_limit'] >= 41) & (df['speed_limit'] <= 50),
            (df['speed_limit'] >= 51) & (df['speed_limit'] <= 80),
            (df['speed_limit'] > 80)
        ],
        ['Low', 'Med', 'High', 'Very High'],
        default=np.nan
    ),
    np.select(
        [
            (df['speed_limit'] > 0) & (df['speed_limit'] <= 40),
            (df['speed_limit'] >= 41) & (df['speed_limit'] <= 60),
            (df['speed_limit'] >= 61) & (df['speed_limit'] <= 80),
            (df['speed_limit'] > 80)
        ],
        ['Low', 'Med', 'High', 'Very High'],
        default=np.nan
    )
)

# change 'nan' value to np.nan
df['speed_limit'] = df['speed_limit'].replace('nan', np.nan)            # КОСТЫЛЬ
# check if there are any NaN values in speed_limit
print(df['speed_limit'].isna().sum())


1485


In [117]:
print(df['speed_limit'].unique())

['Very High' 'High' 'Med' nan 'Low']


In [118]:
# drop dayweek, age, christmas_period and easter_period columns
df.drop(columns=['dayweek', 'age', 'christmas_period', 'easter_period'], inplace=True)
print(df.head())

   crash_id  victim_number state  month  year      time crash_type  \
0  20241115              1   NSW     12  2024  Not Rush     Single   
1  20241125              1   NSW     12  2024  Not Rush     Single   
2  20246013              1   Tas     12  2024  Not Rush   Multiple   
3  20241002              1   NSW     12  2024  Not Rush   Multiple   
4  20242261              1   Vic     12  2024  Not Rush   Multiple   

  bus_involvement heavy_rigid_truck_involvement articulated_truck_involvement  \
0              No                            No                            No   
1              No                            No                            No   
2              No                            No                            No   
3              No                            No                            No   
4             NaN                           NaN                           NaN   

   ...  road_user  gender          remoteness_areas  \
0  ...     Driver    Male  Inner Regi

In [119]:
# Save the cleaned DataFrame to a new Excel file
output_file_path = "bitre_fatalities_cleaned2.xlsx"
df.to_excel(output_file_path, index=False)
print(f"Cleaned data saved to {output_file_path}")

Cleaned data saved to bitre_fatalities_cleaned2.xlsx


## Population fact table

you are required to utilise at least one of the following datasets: Dwelling Count Data or Population Data. You may choose to incorporate both of these additional datasets if desired.

In [120]:
file_path = "Population.xlsx"
population = pd.read_excel(file_path, sheet_name="Table 1", skiprows=5) # Can be improved
print(population.head())

  Unnamed: 0             Unnamed: 1   2001   2002   2003   2004   2005   2006  \
0   LGA code  Local Government Area    no.    no.    no.    no.    no.    no.   
1      10050                 Albury  45265  45816  46180  46505  47004  47566   
2      10180               Armidale  27906  27774  27610  27410  27350  27377   
3      10250                Ballina  37856  38417  38870  39120  39305  39537   
4      10300              Balranald   2751   2703   2661   2596   2545   2507   

    2007   2008  ...   2014   2015   2016   2017   2018   2019   2020   2021  \
0    no.    no.  ...    no.    no.    no.    no.    no.    no.    no.    no.   
1  48140  48518  ...  50990  51486  52171  53056  53922  54657  55466  56067   
2  27468  27788  ...  29015  29160  29310  29519  29631  29701  29600  29332   
3  39824  40020  ...  41881  42336  42993  43652  44385  44997  45663  46196   
4   2473   2433  ...   2376   2364   2330   2338   2308   2287   2257   2208   

    2022   2023  
0    no.    no

In [121]:
# Clean the columnnames
year_colnames = population.columns[2:].tolist()  # Get the first row for year column names
lga_colnames = population.iloc[0, :2].tolist()  # Get the first two columns for LGA names

# change 'local_government_area' to 'lga_name'
lga_colnames[1] = 'lga_name'


for i in range(len(lga_colnames)):
    lga_colnames[i] = lga_colnames[i].strip().lower().replace(' ', '_').replace('/', '_')
population.columns = lga_colnames + year_colnames  # Combine the two lists
population = population[1:-1].reset_index(drop=True)  # Skip the first row and the last row with '© Commonwealth of Australia'
population.head()
population.tail()

,lga_code,lga_name,2001,2002,2003,2004,2005,2006,2007,2008,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
543,74680,West Daly,2543,2612,2674,2752,2864,2986,3023,3151,...,3587,3598,3601,3536,3476,3427,3422,3422,3434,3426
544,79399,Unincorporated NT,7027,7072,7058,7128,7447,7664,7879,7995,...,7970,7112,7065,7034,7023,7102,7263,7420,7571,7713
545,89399,Unincorporated ACT,321538,324627,327357,328940,331399,335170,342644,348368,...,388799,395813,403104,415046,426081,435730,444903,452508,456915,466566
546,99399,Unincorp. Other Territories,542,464,441,428,413,386,370,370,...,361,367,2159,2243,2324,2382,2437,2530,2518,2516
547,NaN,Total Australia,19274701,19495210,19720737,19932722,20176844,20450966,20827622,21249199,...,23475686,23815995,24190907,24592588,24963258,25334826,25649248,25685412,26014399,26648878


In [123]:
# Transform population into long format
population_long = population.melt(
    id_vars = ["lga_code", "lga_name"],
    var_name = "year",
    value_name = "population"
)
population_long




,lga_code,lga_name,year,population
0,10050,Albury,2001,45265
1,10180,Armidale,2001,27906
2,10250,Ballina,2001,37856
3,10300,Balranald,2001,2751
4,10470,Bathurst,2001,35504
...,...,...,...,...
12599,74680,West Daly,2023,3426
12600,79399,Unincorporated NT,2023,7713
12601,89399,Unincorporated ACT,2023,466566
12602,99399,Unincorp. Other Territories,2023,2516


## Designing dimention tables

In [125]:
print(df.columns)

Index(['crash_id', 'victim_number', 'state', 'month', 'year', 'time',
       'crash_type', 'bus_involvement', 'heavy_rigid_truck_involvement',
       'articulated_truck_involvement', 'speed_limit', 'road_user', 'gender',
       'remoteness_areas', 'sa4_name', 'lga_name', 'road_type', 'age_group',
       'day_of_week', 'time_of_day', 'holiday'],
      dtype='object')


In [136]:
# create location_dimension table
location_dim = df[['state', 'lga_name']].drop_duplicates()

location_dim

# add lga_code to location_dim from population_long
location_dim = location_dim.merge(population_long[['lga_name', 'lga_code']].drop_duplicates(), on='lga_name', how='left')

# move lga_code to the front
cols = location_dim.columns.tolist()
cols.insert(0, cols.pop(cols.index('lga_code')))
location_dim = location_dim[cols]

# check if there are lga_name != NaN and lge_code = NaN
print(location_dim[location_dim['lga_name'].notna() & location_dim['lga_code'].isna()])

    lga_code state                               lga_name
3        NaN   NSW                      Armidale Regional
21       NaN   NSW                   Mid-Western Regional
42       NaN   NSW                          Central Coast
43       NaN   NSW                      Tamworth Regional
51       NaN   NSW                         Dubbo Regional
93       NaN   NSW           Queanbeyan-Palerang Regional
94       NaN   NSW                  Snowy Monaro Regional
111      NaN   NSW                           Campbelltown
153      NaN   Vic                               Moreland
171      NaN   NSW                         Unincorporated
179      NaN   NSW                      Bathurst Regional
200      NaN   Tas                            Break O'Day
229      NaN   NSW                                Bayside
469      NaN    SA  Anangu Pitjantjatjara Yunkunytjatjara
480      NaN   NSW                               Nambucca


In [137]:
# create date_dimension table
date_dim = df[['year', 'month']].drop_duplicates()
date_dim['dateID'] = pd.to_datetime(date_dim['year'].astype(str) + '-' + date_dim['month'].astype(str), format='%Y-%m').dt.strftime('%Y%m')

# move dateID to the front
cols = date_dim.columns.tolist()
cols.insert(0, cols.pop(cols.index('dateID')))
date_dim = date_dim[cols]

date_dim

,dateID,year,month
0,202412,2024,12
113,202411,2024,11
244,202410,2024,10
359,202409,2024,9
454,202408,2024,8
...,...,...,...
55778,198905,1989,5
55991,198904,1989,4
56179,198903,1989,3
56435,198902,1989,2
